
# STEP 34A — Cohort Merger

Notebook นี้รวมไฟล์:

- `evaluation_summary_with_labels.csv`
- `oasis_cross-sectional*.xlsx`

เพื่อสร้าง cohort สำหรับ Table 1

เกณฑ์ผ่าน:

- Labelled subjects = 212
- CN = 124
- AD = 88
- Matched = 212
- Unmatched = 0
- Duplicate IDs = 0


In [ ]:

from pathlib import Path
import json, re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

ROOTS = [
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    Path.cwd()
]
ROOT = next((p for p in ROOTS if p.exists()), Path.cwd())
INPUT_DIR = ROOT / "34_Table_Generator_Input"
OUTPUT_DIR = ROOT / "34A_Cohort_Merger_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_FILE = INPUT_DIR / "evaluation_summary_with_labels.csv"
CLINICAL_FILES = sorted(
    list(INPUT_DIR.glob("oasis_cross-sectional*.xlsx"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.xls"))
    + list(INPUT_DIR.glob("oasis_cross-sectional*.csv"))
)
if not PREDICTION_FILE.exists():
    raise FileNotFoundError(PREDICTION_FILE)
if not CLINICAL_FILES:
    raise FileNotFoundError("No OASIS clinical spreadsheet found")

CLINICAL_FILE = CLINICAL_FILES[0]
print("Prediction:", PREDICTION_FILE)
print("Clinical  :", CLINICAL_FILE)
print("Output    :", OUTPUT_DIR)


In [ ]:

def load_table(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path)

def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for x in candidates:
        if x.lower() in lookup:
            return lookup[x.lower()]
    return None

def normalize_subject_id(v):
    if pd.isna(v):
        return np.nan
    text = str(v).strip().upper().replace("-", "_").replace(" ", "_")
    text = re.sub(r"_+", "_", text)
    m = re.search(r"(OAS1_\d{4}_MR\d+)", text)
    return m.group(1) if m else text

def normalize_label(v):
    if pd.isna(v):
        return np.nan
    text = str(v).strip().upper()
    if text in {"CN","0","CONTROL","NORMAL","COGNITIVELY NORMAL","FALSE"}:
        return 0
    if text in {"AD","1","ALZHEIMER","ALZHEIMER'S DISEASE","DEMENTED","TRUE"}:
        return 1
    try:
        x = float(text)
        if x in (0,1): return int(x)
    except:
        pass
    return np.nan


In [ ]:

pred_raw = load_table(PREDICTION_FILE)
clinical_raw = load_table(CLINICAL_FILE)

print("Prediction shape:", pred_raw.shape)
print("Clinical shape  :", clinical_raw.shape)
print("Prediction columns:", list(pred_raw.columns))
print("Clinical columns  :", list(clinical_raw.columns))


In [ ]:

pred_id_col = first_existing(pred_raw.columns, ["case_id","subject_key","subject_id","id","subject"])
true_col = first_existing(pred_raw.columns, [
    "ground_truth","ground_truth_original","ground_truth_derived",
    "clinical_labels_observed","clinical_label","true_label",
    "label","diagnosis","class","target","y_true"
])
prob_col = first_existing(pred_raw.columns, [
    "probability_positive","subject_probability","subject_level_probability",
    "ad_probability","predicted_ad_probability","mean_probability",
    "probability","prediction_probability","y_prob","score"
])

if pred_id_col is None or true_col is None or prob_col is None:
    raise KeyError(f"Missing required columns: id={pred_id_col}, label={true_col}, prob={prob_col}")

pred = pred_raw.copy()
pred["_subject_id"] = pred[pred_id_col].map(normalize_subject_id)
pred["_y_true"] = pred[true_col].map(normalize_label)
pred["_y_prob"] = pd.to_numeric(pred[prob_col], errors="coerce")

pred = pred[
    pred["_subject_id"].notna()
    & pred["_y_true"].notna()
    & pred["_y_prob"].notna()
    & pred["_y_prob"].between(0,1)
].copy()

pred["_y_true"] = pred["_y_true"].astype(int)
pred["_group"] = pred["_y_true"].map({0:"CN",1:"AD"})

duplicate_pred = pred[pred["_subject_id"].duplicated(keep=False)].copy()
pred_unique = pred.drop_duplicates("_subject_id", keep="first").copy()

print("ID column   :", pred_id_col)
print("Label column:", true_col)
print("Prob column :", prob_col)
print(pred_unique["_group"].value_counts())
print("Total:", len(pred_unique))


In [ ]:

clinical_id_col = first_existing(clinical_raw.columns, ["ID","subject_id","subject","subject_key","case_id"])
if clinical_id_col is None:
    raise KeyError("No clinical ID column found")

clinical = clinical_raw.copy()
clinical["_subject_id"] = clinical[clinical_id_col].map(normalize_subject_id)
duplicate_clinical = clinical[clinical["_subject_id"].duplicated(keep=False)].copy()
clinical_unique = clinical.drop_duplicates("_subject_id", keep="first").copy()

merged = pred_unique.merge(
    clinical_unique,
    on="_subject_id",
    how="left",
    suffixes=("_pred","_clinical"),
    indicator=True
)

matched = merged[merged["_merge"] == "both"].copy()
unmatched = merged[merged["_merge"] != "both"].copy()

print(merged["_merge"].value_counts())
print("Matched:", len(matched))
print("Unmatched:", len(unmatched))


In [ ]:

audit = pd.DataFrame({
    "Check": [
        "Labelled subjects","CN subjects","AD subjects",
        "Matched clinical records","Unmatched subjects",
        "Duplicate prediction IDs","Duplicate clinical IDs"
    ],
    "Value": [
        len(pred_unique),
        int((pred_unique["_group"]=="CN").sum()),
        int((pred_unique["_group"]=="AD").sum()),
        len(matched),
        len(unmatched),
        int(duplicate_pred["_subject_id"].nunique()),
        int(duplicate_clinical["_subject_id"].nunique())
    ],
    "Expected": [212,124,88,212,0,0,0]
})
audit["Pass"] = audit["Value"] == audit["Expected"]
display(audit)


In [ ]:

matched.to_excel(OUTPUT_DIR/"Merged_Cohort_212.xlsx", index=False)
matched.to_csv(OUTPUT_DIR/"Merged_Cohort_212.csv", index=False, encoding="utf-8-sig")
audit.to_excel(OUTPUT_DIR/"Cohort_Merge_Audit.xlsx", index=False)
unmatched.to_excel(OUTPUT_DIR/"Unmatched_Subjects.xlsx", index=False)
duplicate_pred.to_excel(OUTPUT_DIR/"Duplicate_Prediction_IDs.xlsx", index=False)
duplicate_clinical.to_excel(OUTPUT_DIR/"Duplicate_Clinical_IDs.xlsx", index=False)

manifest = {
    "prediction_file": str(PREDICTION_FILE),
    "clinical_file": str(CLINICAL_FILE),
    "prediction_id_column": pred_id_col,
    "clinical_id_column": clinical_id_col,
    "ground_truth_column": true_col,
    "probability_column": prob_col,
    "labelled_subjects": int(len(pred_unique)),
    "CN": int((pred_unique["_group"]=="CN").sum()),
    "AD": int((pred_unique["_group"]=="AD").sum()),
    "matched_subjects": int(len(matched)),
    "unmatched_subjects": int(len(unmatched)),
    "duplicate_prediction_ids": int(duplicate_pred["_subject_id"].nunique()),
    "duplicate_clinical_ids": int(duplicate_clinical["_subject_id"].nunique()),
    "all_checks_passed": bool(audit["Pass"].all())
}
with open(OUTPUT_DIR/"Cohort_Merge_Manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,ensure_ascii=False,indent=2)

print("Saved to:", OUTPUT_DIR)
print("PASS" if audit["Pass"].all() else "STOP: audit failed")



ผลลัพธ์:

- `Merged_Cohort_212.xlsx`
- `Merged_Cohort_212.csv`
- `Cohort_Merge_Audit.xlsx`
- `Unmatched_Subjects.xlsx`
- `Duplicate_Prediction_IDs.xlsx`
- `Duplicate_Clinical_IDs.xlsx`
- `Cohort_Merge_Manifest.json`
